In [ ]:
import copy
import heapq

def evaluate_board(board):
    values = {'P':1, 'N':3, 'B':3, 'R':5, 'Q':9, 'K':0,
              'p':-1, 'n':-3, 'b':-3, 'r':-5, 'q':-9, 'k':0}
    score = 0
    for row in board:
        for piece in row:
            score += values.get(piece, 0)
    return score

def generate_moves(board, white_to_move=True):
    moves = []
    direction = -1 if white_to_move else 1
    pawn = 'P' if white_to_move else 'p'
    for r in range(8):
        for c in range(8):
            if board[r][c] == pawn:
                nr = r + direction
                if 0 <= nr < 8 and board[nr][c] == '.':
                    moves.append((r, c, nr, c))
    return moves

def make_move(board, move):
    r1, c1, r2, c2 = move
    new_board = copy.deepcopy(board)
    new_board[r2][c2] = new_board[r1][c1]
    new_board[r1][c1] = '.'
    return new_board

def beam_search_chess(board, beam_width, depth_limit, white_to_move=True):
    beam = [(-evaluate_board(board), [], board, white_to_move)]
    best_seq = []
    best_score = float('-inf') if white_to_move else float('inf')
    for depth in range(depth_limit):
        candidates = []
        for neg_eval, seq, b, wtm in beam:
            moves = generate_moves(b, wtm)
            if not moves:
                candidates.append((neg_eval, seq, b, not wtm))
                continue
            for move in moves:
                new_b = make_move(b, move)
                score = evaluate_board(new_b)
                new_seq = seq + [move]
                new_eval = -score if wtm else score
                candidates.append((new_eval, new_seq, new_b, not wtm))
        if white_to_move:
            beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])
        else:
            beam = heapq.nlargest(beam_width, candidates, key=lambda x: x[0])
    if white_to_move:
        best = min(beam, key=lambda x: x[0])
    else:
        best = max(beam, key=lambda x: x[0])
    best_seq = best[1]
    best_score = -best[0] if white_to_move else best[0]
    return best_seq, best_score

board = [
    ['.','.','.','.','.','.','.','.'],
    ['P','P','P','P','P','P','P','P'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.'],
    ['p','p','p','p','p','p','p','p'],
    ['.','.','.','.','.','.','.','.']
]
beam_width = 2
depth_limit = 3
best_seq, best_score = beam_search_chess(board, beam_width, depth_limit, white_to_move=True)
print("Best move sequence:", best_seq)
print("Evaluation score:", best_score)

Best move sequence: [(1, 0, 0, 0), (6, 0, 7, 0), (1, 1, 0, 1)]
Evaluation score: 0


In [3]:
import random
import math

def total_distance(route, points):
    dist = 0
    for i in range(len(route)):
        x1, y1 = points[route[i-1]]
        x2, y2 = points[route[i]]
        dist += math.sqrt((x2-x1)**2 + (y2-y1)**2)
    return dist

def hill_climb_tsp(points, max_iter=1000):
    n = len(points)
    route = list(range(n))
    random.shuffle(route)
    best_dist = total_distance(route, points)
    best_route = route[:]
    for _ in range(max_iter):
        i, j = random.sample(range(n), 2)
        neighbor = best_route[:]
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
        neighbor_dist = total_distance(neighbor, points)
        if neighbor_dist < best_dist:
            best_dist = neighbor_dist
            best_route = neighbor[:]
    return best_route, best_dist

points = [(0,0), (1,5), (5,2), (6,6), (8,3)]
route, dist = hill_climb_tsp(points, max_iter=2000)
print("Optimized route:", route)
print("Total distance:", dist)

Optimized route: [0, 1, 3, 4, 2]
Total distance: 22.35103276995244


In [ ]:
import random
import math

def total_distance(route, points):
    dist = 0
    for i in range(len(route)):
        x1, y1 = points[route[i-1]]
        x2, y2 = points[route[i]]
        dist += math.sqrt((x2-x1)**2 + (y2-y1)**2)
    return dist

def create_population(size, n):
    return [random.sample(range(n), n) for _ in range(size)]

def crossover(parent1, parent2):
    n = len(parent1)
    start, end = sorted(random.sample(range(n), 2))
    child = [None]*n
    child[start:end+1] = parent1[start:end+1]
    fill = [city for city in parent2 if city not in child]
    idx = 0
    for i in range(n):
        if child[i] is None:
            child[i] = fill[idx]
            idx += 1
    return child

def mutate(route, rate=0.1):
    route = route[:]
    for i in range(len(route)):
        if random.random() < rate:
            j = random.randint(0, len(route)-1)
            route[i], route[j] = route[j], route[i]
    return route

def select(population, points, k=3):
    selected = random.sample(population, k)
    selected.sort(key=lambda r: total_distance(r, points))
    return selected[0]

def genetic_tsp(points, pop_size=100, generations=500, mutation_rate=0.1):
    n = len(points)
    population = create_population(pop_size, n)
    best_route = min(population, key=lambda r: total_distance(r, points))
    best_dist = total_distance(best_route, points)
    for gen in range(generations):
        new_pop = []
        for _ in range(pop_size):
            parent1 = select(population, points)
            parent2 = select(population, points)
            child = crossover(parent1, parent2)
            child = mutate(child, mutation_rate)
            new_pop.append(child)
        population = new_pop
        current_best = min(population, key=lambda r: total_distance(r, points))
        current_dist = total_distance(current_best, points)
        if current_dist < best_dist:
            best_dist = current_dist
            best_route = current_best[:]
    return best_route, best_dist

points = [(random.randint(0,20), random.randint(0,20)) for _ in range(10)]
route, dist = genetic_tsp(points, pop_size=100, generations=500, mutation_rate=0.1)
print("Cities:", points)
print("Best route:", route)
print("Total distance:", dist)

Cities: [(13, 20), (11, 0), (7, 8), (6, 0), (0, 5), (20, 4), (12, 6), (13, 4), (5, 16), (9, 11)]
Best route: [9, 0, 8, 4, 3, 1, 5, 7, 6, 2]
Total distance: 71.76206722319087


In [5]:
import heapq
import copy

def evaluate_allocation(allocation, jobs, n_proc):
    loads = [0]*n_proc
    for proc, job_idxs in enumerate(allocation):
        for idx in job_idxs:
            loads[proc] += jobs[idx][0]
    return max(loads)

def beam_search_job_allocation(jobs, n_proc, beam_width=3, depth_limit=None):
    n_jobs = len(jobs)
    if depth_limit is None:
        depth_limit = n_jobs
    initial = [[] for _ in range(n_proc)]
    beam = [(0, initial, set())]
    for depth in range(depth_limit):
        candidates = []
        for max_load, alloc, assigned in beam:
            unassigned = [i for i in range(n_jobs) if i not in assigned]
            if not unassigned:
                candidates.append((max_load, alloc, assigned))
                continue
            unassigned.sort(key=lambda i: -jobs[i][1])
            job_idx = unassigned[0]
            for proc in range(n_proc):
                new_alloc = copy.deepcopy(alloc)
                new_alloc[proc].append(job_idx)
                new_assigned = assigned | {job_idx}
                loads = [sum(jobs[idx][0] for idx in new_alloc[p]) for p in range(n_proc)]
                new_max_load = max(loads)
                candidates.append((new_max_load, new_alloc, new_assigned))
        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])
    best = min(beam, key=lambda x: x[0])
    return best[1], best[0]

jobs = [
    (5, 2), (3, 1), (2, 3), (7, 2), (4, 1)
]
n_proc = 2
allocation, max_load = beam_search_job_allocation(jobs, n_proc, beam_width=3)
print("Task allocation (job indices per processor):", allocation)
print("Maximum processor load:", max_load)

Task allocation (job indices per processor): [[2, 0, 1], [3, 4]]
Maximum processor load: 11
